# MLP Experience Replay
Train an MLPClassifier with year-wise incremental scaling and experience replay.

In [ ]:
import copy
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

## 1. Load Data and Splits

In [ ]:
ds_path = Path('.') / 'training_data_with_features_plus_monthly_indices.zarr'
print(f'Loading data from {ds_path}...')
ds = xr.open_dataset(ds_path, engine='zarr')
print('Data loaded')

split_path = Path('.') / 'data_split.npz'
print(f'Loading split from {split_path}...')
split_data = np.load(split_path)
train_pixel_indices = split_data['train_pixel_indices']
val_pixel_indices = split_data['val_pixel_indices']
test_pixel_indices = split_data['test_pixel_indices']
print('Split loaded')

print('Dataset info:')
print(f'  Total pixels: {len(ds.pixel)}')
print(f'  Total years: {len(ds.year)}')
print(f'  Train pixels: {len(train_pixel_indices)}')
print(f'  Val pixels: {len(val_pixel_indices)}')
print(f'  Test pixels: {len(test_pixel_indices)}')

## 2. Feature Engineering

In [ ]:
def prepare_features_for_year(ds, pixel_indices, year_idx, scaler=None, scaler_mode='auto'):
    """Extract and clean features for one year. Year 0 is skipped by design."""
    if year_idx == 0:
        return np.empty((0, 0)), np.empty((0,)), scaler

    valid_scaler_modes = {'auto', 'fit', 'partial_fit', 'transform', 'none'}
    if scaler_mode not in valid_scaler_modes:
        raise ValueError(f"Invalid scaler_mode '{scaler_mode}'. Valid options: {sorted(valid_scaler_modes)}")

    s2_all_years = ds['s2_bands'].isel(pixel=pixel_indices).values
    s2_mean_per_pixel = np.nanmean(s2_all_years, axis=1)

    ds_subset = ds.isel(pixel=pixel_indices, year=year_idx)
    s2_features = ds_subset['s2_bands'].values
    if np.isnan(s2_features).any():
        s2_features = np.where(np.isnan(s2_features), s2_mean_per_pixel, s2_features)

    dem_features = ds_subset['dem'].values.reshape(-1, 1)
    ndvi_features = ds_subset['ndvi'].values.reshape(-1, 1)
    ndwi_features = ds_subset['ndwi'].values.reshape(-1, 1)

    ds_prev = ds.isel(pixel=pixel_indices, year=year_idx - 1)
    ndvi_last_year = np.where(np.isnan(ds_prev['ndvi'].values.reshape(-1, 1)), 0, ds_prev['ndvi'].values.reshape(-1, 1))
    ndwi_last_year = np.where(np.isnan(ds_prev['ndwi'].values.reshape(-1, 1)), 0, ds_prev['ndwi'].values.reshape(-1, 1))

    required_last_year_bands = ['B04', 'B03', 'B06']
    band_to_idx = {band: i for i, band in enumerate(ds['s2_band'].values)}
    missing_bands = [band for band in required_last_year_bands if band not in band_to_idx]
    if missing_bands:
        raise ValueError(f"Missing required S2 bands for last-year features: {missing_bands}")

    last_year_s2_features = []
    for band in required_last_year_bands:
        band_values = ds_prev['s2_bands'].sel(s2_band=band).values
        band_values = np.where(np.isnan(band_values), 0, band_values)
        last_year_s2_features.append(band_values.reshape(-1, 1))

    # Keep feature order aligned with MLP training notebook.
    features_list = [
        s2_features,
        dem_features,
        ndvi_features,
        ndwi_features,
        ndvi_last_year,
        ndwi_last_year,
        *last_year_s2_features,
    ]

    if 'nbr' in ds.data_vars:
        features_list.append(ds_subset['nbr'].values.reshape(-1, 1))

    if year_idx > 0 and 'ndvi_delta' in ds.data_vars:
        delta_year_idx = year_idx - 1
        ds_delta = ds.isel(pixel=pixel_indices, year=delta_year_idx)
        features_list.append(ds_delta['ndvi_delta'].values.reshape(-1, 1))
        features_list.append(ds_delta['ndwi_delta'].values.reshape(-1, 1))
        if 'nbr_delta' in ds.data_vars:
            features_list.append(ds_delta['nbr_delta'].values.reshape(-1, 1))

    for var_name in ['years_since_last_disturbance', 'log_years_since_last_disturbance', 'ever_disturbed']:
        if var_name in ds.data_vars:
            features_list.append(ds_subset[var_name].values.reshape(-1, 1))

    yearly_index_feature_names = [
        'ndvi_cv_year',
        'ndvi_max_m2m_drop_year',
        'ndvi_max_year',
        'ndvi_min_year',
        'ndvi_std_year',
        'ndwi_cv_year',
        'ndwi_max_m2m_drop_year',
        'ndwi_max_year',
        'ndwi_min_year',
        'ndwi_std_year',
    ]
    for feature_name in yearly_index_feature_names:
        if feature_name in ds.data_vars:
            features_list.append(ds_subset[feature_name].values.reshape(-1, 1))

    X = np.concatenate(features_list, axis=1)
    y = ds_subset['disturbances'].values

    valid_label_mask = np.isin(y, [0, 1])
    X = X[valid_label_mask]
    y = y[valid_label_mask]

    nan_mask = ~np.isnan(X).any(axis=1)
    X_clean = X[nan_mask]
    y_clean = y[nan_mask]

    if len(X_clean) == 0:
        return np.empty((0, X.shape[1] if X.shape[0] > 0 else 0)), np.empty((0,)), scaler

    if scaler_mode == 'none':
        return X_clean, y_clean, scaler

    if scaler is None:
        scaler = StandardScaler()

    if scaler_mode == 'auto':
        if hasattr(scaler, 'mean_'):
            X_clean = scaler.transform(X_clean)
        else:
            X_clean = scaler.fit_transform(X_clean)
    elif scaler_mode == 'fit':
        X_clean = scaler.fit_transform(X_clean)
    elif scaler_mode == 'partial_fit':
        scaler.partial_fit(X_clean)
        X_clean = scaler.transform(X_clean)
    elif scaler_mode == 'transform':
        if not hasattr(scaler, 'mean_'):
            raise ValueError("Scaler must be fitted before using scaler_mode='transform'.")
        X_clean = scaler.transform(X_clean)

    return X_clean, y_clean, scaler

## 3. Initialize Class Weights and MLP

In [ ]:
print('Computing class weights from training data...')
all_train_labels = []
for year_idx in range(1, len(ds.year)):
    _, y_batch, _ = prepare_features_for_year(
        ds,
        train_pixel_indices,
        year_idx,
        scaler=None,
        scaler_mode='none',
    )
    if len(y_batch) > 0:
        all_train_labels.extend(y_batch)

all_train_labels = np.array(all_train_labels)
all_train_labels = all_train_labels[~np.isnan(all_train_labels)].astype(int)
all_train_labels = all_train_labels[np.isin(all_train_labels, [0, 1])]
if len(all_train_labels) == 0:
    raise ValueError('No valid training labels after filtering.')

classes = np.array([0, 1])
class_weights_array = compute_class_weight('balanced', classes=classes, y=all_train_labels)
class_weight_dict = {classes[i]: class_weights_array[i] for i in range(len(classes))}

print('Class weights:')
print(f"  Class 0: {class_weight_dict[0]:.4f}")
print(f"  Class 1: {class_weight_dict[1]:.4f}")

model = MLPClassifier(
    hidden_layer_sizes=(64,),
    activation='relu',
    alpha=0.0001,
    random_state=42,
    solver='adam',
    learning_rate='adaptive',
    max_iter=1,
    learning_rate_init=0.001,
    warm_start=False,
    verbose=False,
)
print('MLP initialized')

## 4. Replay-Enabled Online Training

In [ ]:
import json
from datetime import datetime

REPLAY_RATIOS = [0.2, 0.3, 0.4, 0.5]
REPLAY_ENABLED = True
CHUNK_SIZE = 50000
MAX_EPOCHS = 15
PATIENCE = 3
MIN_DELTA = 0.0005
REPLAY_RANDOM_STATE = 42

CHECKPOINT_DIR = Path('.') / 'training_checkpoints_mlp_experience_replay'
CHECKPOINT_DIR.mkdir(exist_ok=True)
HISTORY_CHECKPOINT_FILE = CHECKPOINT_DIR / 'mlp_replay_all_training_histories.pkl'
COMPLETION_STATUS_FILE = CHECKPOINT_DIR / 'mlp_replay_completion_status.json'
TRAINING_LOG_FILE = CHECKPOINT_DIR / 'mlp_replay_training_log.txt'

MLP_KWARGS = dict(
    hidden_layer_sizes=(64,),
    activation='relu',
    alpha=0.0001,
    random_state=42,
    solver='adam',
    learning_rate='adaptive',
    max_iter=1,
    learning_rate_init=0.001,
    warm_start=False,
    verbose=False,
)

n_years = len(ds.year)
year_values = ds.year.values


def _empty_history():
    return {
        'year': [],
        'train_accuracy': [],
        'train_precision': [],
        'train_recall': [],
        'train_f1': [],
        'val_accuracy': [],
        'val_precision': [],
        'val_recall': [],
        'val_f1': [],
        'val_roc_auc': [],
        'val_pr_auc': [],
        'replay_pool_size': [],
        'replay_target_size': [],
        'replay_used_size': [],
    }


def _load_training_histories():
    if HISTORY_CHECKPOINT_FILE.exists():
        with open(HISTORY_CHECKPOINT_FILE, 'rb') as f:
            return pickle.load(f)
    return {}


def _save_training_histories(histories):
    with open(HISTORY_CHECKPOINT_FILE, 'wb') as f:
        pickle.dump(histories, f)


def _load_completion_status():
    if COMPLETION_STATUS_FILE.exists():
        with open(COMPLETION_STATUS_FILE, 'r') as f:
            return json.load(f)
    return {'completed_ratios': [], 'completed_years': {}}


def _save_completion_status(status):
    with open(COMPLETION_STATUS_FILE, 'w') as f:
        json.dump(status, f, indent=2)


def _log(msg):
    ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    with open(TRAINING_LOG_FILE, 'a') as f:
        f.write(f'[{ts}] {msg}\n')
    print(msg)


all_training_histories = _load_training_histories()
completion_status = _load_completion_status()

for _rr in REPLAY_RATIOS:
    _rk = f'RR_{_rr:.1f}'
    if _rk not in all_training_histories:
        all_training_histories[_rk] = _empty_history()

print(f'Checkpoint directory: {CHECKPOINT_DIR.resolve()}')
print(f'Configured replay ratios: {REPLAY_RATIOS}')
print(f"Completed ratios from checkpoint: {completion_status.get('completed_ratios', [])}")

for replay_ratio in REPLAY_RATIOS:
    ratio_key = f'RR_{replay_ratio:.1f}'
    OUTPUT_SUFFIX = f'incremental_scaler_experience_replay_{ratio_key}'
    MODELS_DIR_NAME = f'models_mlp_prevyears_monthly_features_{OUTPUT_SUFFIX}'
    SCALER_FILE_TEMPLATE = f'scaler_year_{{year}}_mlp_prevyears_monthly_features_{OUTPUT_SUFFIX}.pkl'
    MODEL_FILE_TEMPLATE = f'model_year_{{year}}_mlp_prevyears_monthly_features_{OUTPUT_SUFFIX}.pkl'
    FINAL_SCALER_FILENAME = f'scaler_final_mlp_prevyears_monthly_features_{OUTPUT_SUFFIX}.pkl'
    FINAL_MODEL_FILENAME = f'mlp_classifier_model_prevyears_monthly_features_{OUTPUT_SUFFIX}.pkl'
    HISTORY_FILENAME = f'mlp_classifier_history_prevyears_monthly_features_{OUTPUT_SUFFIX}.csv'

    models_dir = Path('.') / MODELS_DIR_NAME
    models_dir.mkdir(exist_ok=True)

    if ratio_key in completion_status.get('completed_ratios', []):
        _log(f'{ratio_key}: already completed, skipping.')
        continue

    training_history = all_training_histories[ratio_key]
    completed_years_for_ratio = list(training_history['year'])

    if completed_years_for_ratio:
        last_year = int(completed_years_for_ratio[-1])
        year_model_resume_path = models_dir / MODEL_FILE_TEMPLATE.format(year=last_year)
        year_scaler_resume_path = models_dir / SCALER_FILE_TEMPLATE.format(year=last_year)

        if year_model_resume_path.exists() and year_scaler_resume_path.exists():
            with open(year_model_resume_path, 'rb') as f:
                model = pickle.load(f)
            with open(year_scaler_resume_path, 'rb') as f:
                incremental_scaler = copy.deepcopy(pickle.load(f))
            _log(f'Resuming {ratio_key} from year {last_year}.')
        else:
            _log(
                f'Artifacts missing for {ratio_key} year {last_year}; restarting this ratio from scratch.'
            )
            training_history = _empty_history()
            all_training_histories[ratio_key] = training_history
            completed_years_for_ratio = []
            model = MLPClassifier(**MLP_KWARGS)
            incremental_scaler = StandardScaler()
    else:
        model = MLPClassifier(**MLP_KWARGS)
        incremental_scaler = StandardScaler()
        _log(f'Starting {ratio_key} from scratch.')

    year_to_scaler_checkpoint = {}
    replay_rng = np.random.default_rng(REPLAY_RANDOM_STATE + int(replay_ratio * 100))

    for year_idx in tqdm(range(1, n_years), desc=f'{ratio_key} - Training by year'):
        year_val = int(year_values[year_idx])

        if year_val in completed_years_for_ratio:
            year_scaler_path = models_dir / SCALER_FILE_TEMPLATE.format(year=year_val)
            if year_scaler_path.exists():
                with open(year_scaler_path, 'rb') as f:
                    incremental_scaler = copy.deepcopy(pickle.load(f))
                    year_to_scaler_checkpoint[year_val] = copy.deepcopy(incremental_scaler)
            continue

        X_train_batch, y_train_batch, incremental_scaler = prepare_features_for_year(
            ds,
            train_pixel_indices,
            year_idx,
            scaler=incremental_scaler,
            scaler_mode='partial_fit',
        )
        X_val_batch, y_val_batch, _ = prepare_features_for_year(
            ds,
            val_pixel_indices,
            year_idx,
            scaler=incremental_scaler,
            scaler_mode='transform',
        )

        if len(X_train_batch) == 0 or len(X_val_batch) == 0:
            print(f'{ratio_key} year {year_val}: skipped (empty after filtering)')
            continue

        scaler_checkpoint = copy.deepcopy(incremental_scaler)
        year_to_scaler_checkpoint[year_val] = scaler_checkpoint
        year_scaler_path = models_dir / SCALER_FILE_TEMPLATE.format(year=year_val)
        with open(year_scaler_path, 'wb') as f:
            pickle.dump(scaler_checkpoint, f)

        n_samples = len(X_train_batch)
        replay_target_size = int(n_samples * replay_ratio) if REPLAY_ENABLED else 0

        replay_X_parts = []
        replay_y_parts = []
        if REPLAY_ENABLED and year_idx > 1:
            for past_year_idx in range(1, year_idx):
                X_past, y_past, _ = prepare_features_for_year(
                    ds,
                    train_pixel_indices,
                    past_year_idx,
                    scaler=incremental_scaler,
                    scaler_mode='transform',
                )
                if len(X_past) > 0:
                    replay_X_parts.append(X_past)
                    replay_y_parts.append(y_past)

        if replay_X_parts:
            X_replay_pool = np.vstack(replay_X_parts)
            y_replay_pool = np.concatenate(replay_y_parts)
        else:
            X_replay_pool = np.empty((0, X_train_batch.shape[1]), dtype=X_train_batch.dtype)
            y_replay_pool = np.empty((0,), dtype=y_train_batch.dtype)

        replay_pool_size = len(y_replay_pool)
        replay_used_size = min(replay_target_size, replay_pool_size) if REPLAY_ENABLED else 0

        if hasattr(model, 'n_features_in_') and int(model.n_features_in_) != int(X_train_batch.shape[1]):
            raise ValueError(
                f'Feature count mismatch at year {year_val}: model expects {model.n_features_in_}, got {X_train_batch.shape[1]}'
            )

        best_val_pr_auc = -np.inf
        patience_counter = 0
        best_model_state = None

        for epoch in range(MAX_EPOCHS):
            if replay_used_size > 0:
                replay_indices = replay_rng.choice(replay_pool_size, size=replay_used_size, replace=False)
                X_replay_sampled = X_replay_pool[replay_indices]
                y_replay_sampled = y_replay_pool[replay_indices]
                X_combined = np.concatenate([X_train_batch, X_replay_sampled], axis=0)
                y_combined = np.concatenate([y_train_batch, y_replay_sampled], axis=0)
            else:
                X_combined = X_train_batch
                y_combined = y_train_batch

            combined_n_samples = len(X_combined)
            shuffle_idx = np.random.permutation(combined_n_samples)
            X_train_shuffled = X_combined[shuffle_idx]
            y_train_shuffled = y_combined[shuffle_idx]
            n_chunks = max(1, int(np.ceil(combined_n_samples / CHUNK_SIZE)))

            for chunk_idx in range(n_chunks):
                start_idx = chunk_idx * CHUNK_SIZE
                end_idx = min(start_idx + CHUNK_SIZE, combined_n_samples)
                X_chunk = X_train_shuffled[start_idx:end_idx]
                y_chunk = y_train_shuffled[start_idx:end_idx]
                sample_weights_chunk = np.array([class_weight_dict[int(label)] for label in y_chunk])
                model.partial_fit(X_chunk, y_chunk, classes=classes, sample_weight=sample_weights_chunk)

            y_val_proba = model.predict_proba(X_val_batch)[:, 1]
            val_pr_auc = average_precision_score(y_val_batch, y_val_proba) if len(np.unique(y_val_batch)) > 1 else np.nan

            if np.isnan(val_pr_auc):
                continue

            if val_pr_auc > best_val_pr_auc + MIN_DELTA:
                best_val_pr_auc = val_pr_auc
                patience_counter = 0
                best_model_state = {
                    'coefs': [w.copy() for w in model.coefs_],
                    'intercepts': [b.copy() for b in model.intercepts_],
                    'n_layers_': model.n_layers_,
                    'n_outputs_': getattr(model, 'n_outputs_', None),
                    'out_activation_': getattr(model, 'out_activation_', None),
                }
            else:
                patience_counter += 1
                if patience_counter >= PATIENCE:
                    if best_model_state is not None:
                        model.coefs_ = [w.copy() for w in best_model_state['coefs']]
                        model.intercepts_ = [b.copy() for b in best_model_state['intercepts']]
                        model.n_layers_ = best_model_state['n_layers_']
                        if best_model_state['n_outputs_'] is not None:
                            model.n_outputs_ = best_model_state['n_outputs_']
                        if best_model_state['out_activation_'] is not None:
                            model.out_activation_ = best_model_state['out_activation_']
                    break

        y_train_pred = model.predict(X_train_batch)
        y_val_pred = model.predict(X_val_batch)
        y_val_proba = model.predict_proba(X_val_batch)[:, 1]

        train_acc = accuracy_score(y_train_batch, y_train_pred)
        train_prec = precision_score(y_train_batch, y_train_pred, zero_division=0)
        train_rec = recall_score(y_train_batch, y_train_pred, zero_division=0)
        train_f1 = f1_score(y_train_batch, y_train_pred, zero_division=0)

        val_acc = accuracy_score(y_val_batch, y_val_pred)
        val_prec = precision_score(y_val_batch, y_val_pred, zero_division=0)
        val_rec = recall_score(y_val_batch, y_val_pred, zero_division=0)
        val_f1 = f1_score(y_val_batch, y_val_pred, zero_division=0)
        if len(np.unique(y_val_batch)) > 1:
            val_roc_auc = roc_auc_score(y_val_batch, y_val_proba)
            val_pr_auc = average_precision_score(y_val_batch, y_val_proba)
        else:
            val_roc_auc = np.nan
            val_pr_auc = np.nan

        training_history['year'].append(year_val)
        training_history['train_accuracy'].append(train_acc)
        training_history['train_precision'].append(train_prec)
        training_history['train_recall'].append(train_rec)
        training_history['train_f1'].append(train_f1)
        training_history['val_accuracy'].append(val_acc)
        training_history['val_precision'].append(val_prec)
        training_history['val_recall'].append(val_rec)
        training_history['val_f1'].append(val_f1)
        training_history['val_roc_auc'].append(val_roc_auc)
        training_history['val_pr_auc'].append(val_pr_auc)
        training_history['replay_pool_size'].append(int(replay_pool_size))
        training_history['replay_target_size'].append(int(replay_target_size))
        training_history['replay_used_size'].append(int(replay_used_size))

        year_model_path = models_dir / MODEL_FILE_TEMPLATE.format(year=year_val)
        with open(year_model_path, 'wb') as f:
            pickle.dump(model, f)

        all_training_histories[ratio_key] = training_history
        _save_training_histories(all_training_histories)
        completion_status.setdefault('completed_years', {})[ratio_key] = list(training_history['year'])
        _save_completion_status(completion_status)

        _log(
            f'Checkpoint saved: {ratio_key} year {year_val} '
            f'(Val PR-AUC={val_pr_auc:.4f}, replay={replay_used_size:,}/{replay_pool_size:,})'
        )

        print(
            f"{ratio_key} year {year_val}: Train F1={train_f1:.3f}, Val F1={val_f1:.3f}, "
            f"Val PR-AUC={val_pr_auc:.3f}, replay_used={replay_used_size:,}/{replay_pool_size:,}"
        )

    if hasattr(incremental_scaler, 'mean_'):
        final_scaler_path = models_dir / FINAL_SCALER_FILENAME
        with open(final_scaler_path, 'wb') as f:
            pickle.dump(incremental_scaler, f)
        print(f'{ratio_key}: final scaler saved: {final_scaler_path.name}')

    final_model_path = Path('.') / FINAL_MODEL_FILENAME
    with open(final_model_path, 'wb') as f:
        pickle.dump(model, f)
    print(f'{ratio_key}: final model saved: {final_model_path.name}')

    history_df_ratio = pd.DataFrame(training_history).sort_values('year').reset_index(drop=True)
    history_path_ratio = Path('.') / HISTORY_FILENAME
    history_df_ratio.to_csv(history_path_ratio, index=False)
    print(f'{ratio_key}: history saved: {history_path_ratio.name}')

    if ratio_key not in completion_status.get('completed_ratios', []):
        completion_status.setdefault('completed_ratios', []).append(ratio_key)
        _save_completion_status(completion_status)
    _log(f'{ratio_key}: completed.')

year_to_scaler_checkpoint = {}
for _rr in REPLAY_RATIOS:
    _rk = f'RR_{_rr:.1f}'
    if _rk in completion_status.get('completed_ratios', []):
        _suffix = f'incremental_scaler_experience_replay_{_rk}'
        _models_dir = Path('.') / f'models_mlp_prevyears_monthly_features_{_suffix}'
        _scaler_tmpl = f'scaler_year_{{year}}_mlp_prevyears_monthly_features_{_suffix}.pkl'
        _years = completion_status.get('completed_years', {}).get(_rk, [])
        for _y in _years:
            _p = _models_dir / _scaler_tmpl.format(year=int(_y))
            if _p.exists():
                with open(_p, 'rb') as f:
                    year_to_scaler_checkpoint[int(_y)] = pickle.load(f)

print('Replay ratio sweep complete.')

## 5. Save Training History

In [ ]:
combined_rows = []
for replay_ratio in REPLAY_RATIOS:
    ratio_key = f'RR_{replay_ratio:.1f}'
    if ratio_key not in all_training_histories:
        continue
    ratio_history = all_training_histories[ratio_key]
    if not ratio_history['year']:
        continue

    ratio_df = pd.DataFrame(ratio_history).sort_values('year').reset_index(drop=True)
    ratio_df.insert(0, 'replay_ratio', replay_ratio)
    ratio_df.insert(1, 'ratio_key', ratio_key)
    combined_rows.append(ratio_df)

if combined_rows:
    combined_history_df = pd.concat(combined_rows, ignore_index=True)
    combined_history_path = Path('.') / 'mlp_classifier_combined_history_replay_ratios.csv'
    combined_history_df.to_csv(combined_history_path, index=False)
    print(f'Combined history saved: {combined_history_path}')
    print(f'Rows: {len(combined_history_df):,}')
    display(combined_history_df.tail())
else:
    print('No completed replay ratio history to save yet.')